# Binary Classification of Depression Survey Data Using Machine Learning

## Project Introduction
This project involves using machine learning to perform binary classification on a synthetic dataset derived from a depression survey. The goal is to develop a model that accurately predicts a binary target class based on training data.

Key focus areas include data analysis, visualization, and feature engineering to enhance model performance. This challenge offers a chance to refine machine learning skills while working with data that simulates real-world scenarios.

You can access the dataset directly by clicking this link: [Playground Series - Season 4, Episode 11](https://www.kaggle.com/competitions/playground-series-s4e11/data).


In [195]:
# packages
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [196]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
data = pd.concat([train,test])

In [197]:
test.isnull().sum()

id                                           0
Name                                         0
Gender                                       0
Age                                          0
City                                         0
Working Professional or Student              0
Profession                               24632
Academic Pressure                        75033
Work Pressure                            18778
CGPA                                     75034
Study Satisfaction                       75033
Job Satisfaction                         18774
Sleep Duration                               0
Dietary Habits                               5
Degree                                       2
Have you ever had suicidal thoughts ?        0
Work/Study Hours                             0
Financial Stress                             0
Family History of Mental Illness             0
dtype: int64

In [198]:
data.head()

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0.0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1.0
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1.0
3,3,Yuvraj,Male,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1.0
4,4,Rhea,Female,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0.0


In [199]:
data.shape

(234500, 20)

### feature engineering

In [201]:
# "Academic Pressure" and "Study Satisfaction" are relevant only for students, 
# so they are used to fill missing values in "Work Pressure" and "Job Satisfaction" respectively.
data["Work Pressure"].fillna(data["Academic Pressure"], inplace=True)
data["Job Satisfaction"].fillna(data["Study Satisfaction"], inplace=True)

In [202]:
# rename some columns
data = data.rename(columns = {
    "Work Pressure": "Pressure",
    "Job Satisfaction": "Satisfaction",
    "Have you ever had suicidal thoughts ?": "Suicidal Thoughts",
    "Family History of Mental Illness": "Family History"
})

In [203]:
# Set "Profession" to "Student" for students
data.loc[data['Working Professional or Student'] == "Student", "Profession"] = "Student"

In [204]:
data.head()

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Pressure,CGPA,Study Satisfaction,Satisfaction,Sleep Duration,Dietary Habits,Degree,Suicidal Thoughts,Work/Study Hours,Financial Stress,Family History,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0.0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1.0
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,Student,5.0,5.0,8.97,2.0,2.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1.0
3,3,Yuvraj,Male,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1.0
4,4,Rhea,Female,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0.0


In [205]:
data["Gender"]= data["Gender"].map({"Female": 1, "Male":0})
data["Dietary Habits"]= data["Dietary Habits"].map({"Healthy": 1, "Unhealthy":-1, "Moderate":0})
data["Family History"]= data["Family History"].map({"Yes": 1, "No":0})
data["Suicidal Thoughts"]= data["Suicidal Thoughts"].map({"Yes": 1, "No":0})

In [206]:
data.head()

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Pressure,CGPA,Study Satisfaction,Satisfaction,Sleep Duration,Dietary Habits,Degree,Suicidal Thoughts,Work/Study Hours,Financial Stress,Family History,Depression
0,0,Aaradhya,1,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,1.0,BHM,0,1.0,2.0,0,0.0
1,1,Vivan,0,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,-1.0,LLB,1,7.0,3.0,0,1.0
2,2,Yuvraj,0,33.0,Visakhapatnam,Student,Student,5.0,5.0,8.97,2.0,2.0,5-6 hours,1.0,B.Pharm,1,3.0,1.0,0,1.0
3,3,Yuvraj,0,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,0.0,BBA,1,10.0,1.0,1,1.0
4,4,Rhea,1,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,-1.0,BBA,1,9.0,4.0,1,0.0


In [207]:
sleep_map = {
    "Less than 5 hours": 4,
    "5-6 hours": 5,
    "6-7 hours": 6,
    "7-8 hours": 7,
    "8-9 hours": 8,
    "More than 8 hours": 9,
    "3-4 hours": 3,   
    "4-5 hours": 4,
    "1-2 hours": 1,
    "Unhealthy": np.nan,          
    "No": np.nan,                  
    "Sleep_Duration": np.nan,       
    "Meerut": np.nan,               
    "1-3 hours": 2,
    "10-11 hours": 10,
    "9-5": 7,
    "9-6 hours": 8,
    "8-89 hours": np.nan, 
    "Vivan": np.nan,              
    "20-21 hours": np.nan,
    "60-65 hours": np.nan,
    "0": np.nan,
}
data["Sleep Duration"]= data["Sleep Duration"].map(sleep_map)

In [208]:
data.head()

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Pressure,CGPA,Study Satisfaction,Satisfaction,Sleep Duration,Dietary Habits,Degree,Suicidal Thoughts,Work/Study Hours,Financial Stress,Family History,Depression
0,0,Aaradhya,1,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,9.0,1.0,BHM,0,1.0,2.0,0,0.0
1,1,Vivan,0,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,4.0,-1.0,LLB,1,7.0,3.0,0,1.0
2,2,Yuvraj,0,33.0,Visakhapatnam,Student,Student,5.0,5.0,8.97,2.0,2.0,5.0,1.0,B.Pharm,1,3.0,1.0,0,1.0
3,3,Yuvraj,0,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,4.0,0.0,BBA,1,10.0,1.0,1,1.0
4,4,Rhea,1,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5.0,-1.0,BBA,1,9.0,4.0,1,0.0


In [209]:
# drop some columns
data = data.drop(["Name","City","Working Professional or Student","Academic Pressure","Study Satisfaction","Degree","Profession"], axis=1)

In [210]:
# missing values
data["CGPA"].fillna(-1,inplace= True)


In [211]:
data.isnull().sum()

id                       0
Gender                   0
Age                      0
Pressure                33
CGPA                     0
Satisfaction            23
Sleep Duration          70
Dietary Habits          57
Suicidal Thoughts        0
Work/Study Hours         0
Financial Stress         4
Family History           0
Depression           93800
dtype: int64

In [212]:
train2 = data[data["Depression"].notnull()] 

In [213]:
test2 = data[data["Depression"].isnull()] 

In [214]:
train2.isnull().sum()

id                    0
Gender                0
Age                   0
Pressure             21
CGPA                  0
Satisfaction         15
Sleep Duration       44
Dietary Habits       27
Suicidal Thoughts     0
Work/Study Hours      0
Financial Stress      4
Family History        0
Depression            0
dtype: int64

In [215]:
train2=train2.dropna()

In [216]:
train2.shape

(140596, 13)

In [217]:
test2

,id,Gender,Age,Pressure,CGPA,Satisfaction,Sleep Duration,Dietary Habits,Suicidal Thoughts,Work/Study Hours,Financial Stress,Family History,Depression
0,140700,0,53.0,2.0,-1.00,5.0,4.0,0.0,0,9.0,3.0,1,NaN
1,140701,1,58.0,2.0,-1.00,4.0,4.0,0.0,0,6.0,4.0,0,NaN
2,140702,0,53.0,4.0,-1.00,1.0,7.0,0.0,1,12.0,4.0,0,NaN
3,140703,1,23.0,5.0,6.84,1.0,9.0,0.0,1,10.0,4.0,0,NaN
4,140704,0,47.0,5.0,-1.00,5.0,7.0,0.0,1,3.0,4.0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93795,234495,1,49.0,3.0,-1.00,5.0,4.0,0.0,1,2.0,2.0,1,NaN
93796,234496,0,29.0,5.0,-1.00,1.0,7.0,0.0,1,11.0,3.0,1,NaN
93797,234497,0,24.0,1.0,7.51,4.0,7.0,0.0,0,7.0,1.0,0,NaN
93798,234498,1,23.0,4.0,-1.00,2.0,5.0,1.0,1,7.0,5.0,1,NaN


In [221]:
x=train2.drop(['Depression','id'],axis=1)
y=train2[['Depression']]

In [222]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

In [290]:
dt= DecisionTreeClassifier()
dt.fit(x_train,y_train)
pred=dt.predict(x_val)
accuracy_score(y_val,pred)

0.9037695590327169

In [292]:
rc= RandomForestClassifier()
rc.fit(x_train,y_train)
pred= rc.predict(x_val)
accuracy_score(y_val,pred)

C:\Users\zeyil\anaconda3\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.933819345661451

In [296]:
lr= LogisticRegression()
lr.fit(x_train,y_train)
pred= lr.predict(x_val)
accuracy_score(y_val,pred)

C:\Users\zeyil\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
C:\Users\zeyil\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9354196301564722

In [227]:
new_test=test2.drop(['Depression','id'],axis=1)

In [228]:
new_test

,Gender,Age,Pressure,CGPA,Satisfaction,Sleep Duration,Dietary Habits,Suicidal Thoughts,Work/Study Hours,Financial Stress,Family History
0,0,53.0,2.0,-1.00,5.0,4.0,0.0,0,9.0,3.0,1
1,1,58.0,2.0,-1.00,4.0,4.0,0.0,0,6.0,4.0,0
2,0,53.0,4.0,-1.00,1.0,7.0,0.0,1,12.0,4.0,0
3,1,23.0,5.0,6.84,1.0,9.0,0.0,1,10.0,4.0,0
4,0,47.0,5.0,-1.00,5.0,7.0,0.0,1,3.0,4.0,0
...,...,...,...,...,...,...,...,...,...,...,...
93795,1,49.0,3.0,-1.00,5.0,4.0,0.0,1,2.0,2.0,1
93796,0,29.0,5.0,-1.00,1.0,7.0,0.0,1,11.0,3.0,1
93797,0,24.0,1.0,7.51,4.0,7.0,0.0,0,7.0,1.0,0
93798,1,23.0,4.0,-1.00,2.0,5.0,1.0,1,7.0,5.0,1


In [266]:
new_test.isnull().sum()

Gender                0
Age                   0
Pressure             12
CGPA                  0
Satisfaction          8
Sleep Duration       26
Dietary Habits       30
Suicidal Thoughts     0
Work/Study Hours      0
Financial Stress      0
Family History        0
dtype: int64

In [270]:
imputer = SimpleImputer(strategy='mean')

new_test = pd.DataFrame(imputer.fit_transform(new_test), columns=new_test.columns)

In [272]:
predict=lr.predict(new_test)

In [274]:
submission=pd.DataFrame()

In [276]:
submission['id']=test2['id']

In [278]:
submission['Depression']=predict

In [280]:
submission

,id,Depression
0,140700,0.0
1,140701,0.0
2,140702,0.0
3,140703,1.0
4,140704,0.0
...,...,...
93795,234495,0.0
93796,234496,1.0
93797,234497,0.0
93798,234498,1.0


In [284]:
submission["Depression"]=submission["Depression"].astype('int32')

In [288]:
submission.to_csv('submission.csv',index=False)

## Conclusion

In this project, three models were tested for classifying skin lesions:

1. **Decision Tree Classifier** - Accuracy: 90.38%
2. **Random Forest Classifier** - Accuracy: 93.38%
3. **Logistic Regression** - Accuracy: 93.54%

Logistic Regression achieved the highest accuracy at 93.54%, slightly outperforming Random Forest. The Decision Tree had the lowest accuracy at 90.38%. These results highlight the effectiveness of Logistic Regression for skin cancer classification, showing its potential for reliable medical diagnostics.
